## 💾 7. PERSISTENCIA FINAL EN FORMATO PARQUET (CON PURGA PREVENTIVA AUTOMÁTICA)

In [0]:
# ----------------------------------------------------------------------------------------------------
# 💾 7. PERSISTENCIA FINAL EN FORMATO PARQUET (CON PURGA PREVENTIVA AUTOMÁTICA)
# ----------------------------------------------------------------------------------------------------
import os
import shutil

ruta_salida_final = "/Volumes/workspace/default/taller_final/raw/secop_base_analitica.parquet"

print("====================================================================================")
print("🧹 FORCE PURGE: Eliminando caché analítica anterior para evitar bloqueos de Spark...")
print("====================================================================================")

try:
    if os.path.exists(ruta_salida_final):
        shutil.rmtree(ruta_salida_final)
        print("   ✅ Directorio antiguo removido del volumen con éxito.")
    else:
        print("   ℹ️  El directorio está limpio (no había datos previos).")
except Exception as e:
    print(f"   ⚠️  Aviso en limpieza (procediendo con la escritura): {e}")

print("-" * 84)
print(f"💾 Almacenando Master Dataset Enriquecido Reciente en: {ruta_salida_final}")

# Escribimos en limpio sobre un directorio completamente vacío
df_final_taller.write.mode("overwrite").parquet(ruta_salida_final)

print("\n🎉 ACTIVIDAD 2 COMPLETADA CON ÉXITO DE FORMA ABSOLUTA. CAPA SILVER BLINDADA.")
print("=" * 84)

🧹 FORCE PURGE: Eliminando caché analítica anterior para evitar bloqueos de Spark...
   ✅ Directorio antiguo removido del volumen con éxito.
------------------------------------------------------------------------------------
💾 Almacenando Master Dataset Enriquecido Reciente en: /Volumes/workspace/default/taller_final/raw/secop_base_analitica.parquet

🎉 ACTIVIDAD 2 COMPLETADA CON ÉXITO DE FORMA ABSOLUTA. CAPA SILVER BLINDADA.


## 🗺️ 6. CRUCE TERRITORIAL MAESTRO - ESTRATEGIA DE EXTRACCIÓN INVERSA CON DIAGNÓSTICO DE VACÍOS

In [0]:
# ----------------------------------------------------------------------------------------------------
# 🗺️ 6. CRUCE TERRITORIAL MAESTRO - ESTRATEGIA DE EXTRACCIÓN INVERSA CON DIAGNÓSTICO DE VACÍOS
# ----------------------------------------------------------------------------------------------------
print("\n🗺️ 6. Iniciando enriquecimiento financiero final y cruce geográfico con DIVIPOLA...")

df_consolidado_con_pagos = df_consolidado_intermedio \
    .withColumn("total_pagado_acumulado", F.coalesce(F.col("ultimo_monto_ejecutado"), F.col("total_pagado_contrato"), F.lit(0.0)))

# 📊 DIAGNÓSTICO PREVENTIVO: Contamos la basura/vacíos absolutos directo de la fuente antes del join
conteo_ciudad_vacia = df_consolidado_con_pagos.filter(
    F.col("ciudad").isNull() | F.col("ciudad").isin(["", "-", "no definido", "no aplica"])
).count()

conteo_depto_vacio = df_consolidado_con_pagos.filter(
    F.col("departamento").isNull() | F.col("departamento").isin(["", "-", "no definido", "no aplica"])
).count()

def limpieza_extrema_geo(columna):
    text_clean = F.coalesce(F.col(columna), F.lit(""))
    # Removemos ruido estructural masivo que confunde al optimizador de Spark
    text_clean = F.regexp_replace(text_clean, r"(?i),\s*d\.c\.|d\.c\.|distrito\s*capital|distrito|municipio|colombia|san andres de\s*", "")
    text_clean = F.translate(text_clean, "áéíóúÁÉÍÓÚñÑüÜ", "aeiouAEIOUnNuU")
    return F.lower(F.trim(F.regexp_replace(text_clean, r"\s+", " ")))

# 1. Tratamiento preventivo contra datos basura del SECOP ("no definido", vacíos, etc.)
df_consolidado_homologado = df_consolidado_con_pagos.withColumn(
    "ciudad", 
    F.when(F.col("ciudad").rlike("(?i)no definido|no aplica|sin de|colombia|^-$"), F.lit("bogota"))
     .otherwise(F.col("ciudad"))
).withColumn(
    "departamento",
    F.when(F.col("departamento").rlike("(?i)no definido|no aplica|sin de|colombia|^-$"), F.lit("Bogotá"))
     .otherwise(F.col("departamento"))
)

# 2. Configuración macro de departamentos conflictivos
df_consolidado_homologado = df_consolidado_homologado.withColumn(
    "departamento",
    F.when(F.col("departamento").rlike("(?i)Bogotá|Distrito Capital"), F.lit("Bogotá"))
     .when(F.col("departamento").rlike("(?i)San Andrés"), F.lit("Archipiélago de San Andrés, Providencia y Santa Catalina"))
     .when(F.col("departamento").rlike("(?i)Valle del Cauca"), F.lit("Valle del Cauca"))
     .when(F.col("departamento").rlike("(?i)Norte de San|Norte Santander"), F.lit("Norte de Santander"))
     .otherwise(F.col("departamento"))
)

# Preparación de llaves SECOP
df_consolidado_ready = df_consolidado_homologado \
    .withColumn("depto_secop_key", limpieza_extrema_geo("departamento")) \
    .withColumn("muni_secop_key", limpieza_extrema_geo("ciudad"))

df_consolidado_ready = df_consolidado_ready.withColumn(
    "depto_secop_key",
    F.when(F.col("depto_secop_key").rlike("(?i)bogota|distrito|capital|pensiones"), F.lit("bogota d.c."))
     .when(F.col("depto_secop_key").rlike("(?i)san andres"), F.lit("archipielago de san andres"))
     .otherwise(F.col("depto_secop_key"))
).withColumn(
    "muni_secop_key",
    F.when(F.col("depto_secop_key") == "bogota d.c.", F.lit("bogota d.c."))
     .when(F.col("muni_secop_key").rlike("(?i)cucuta"), F.lit("san jose de cucuta"))
     .when(F.col("muni_secop_key").rlike("(?i)cartagena"), F.lit("cartagena de indias"))
     .when(F.col("muni_secop_key").rlike("(?i)quibdo"), F.lit("san francisco de quibdo"))
     .when(F.col("muni_secop_key").rlike("(?i)mocoa"), F.lit("san miguel de mocoa"))
     .when(F.col("muni_secop_key").rlike("(?i)tolu$"), F.lit("santiago de tolu"))
     .otherwise(F.col("muni_secop_key"))
)

# Preparación simétrica de la DIVIPOLA DANE
df_divipola_ready = df_divipola \
    .withColumn("dpto_divipola_key", limpieza_extrema_geo("dpto")) \
    .withColumn("muni_divipola_key", limpieza_extrema_geo("nom_mpio"))

df_divipola_ready = df_divipola_ready.withColumn(
    "dpto_divipola_key",
    F.when(F.col("dpto_divipola_key").rlike("(?i)bogota"), F.lit("bogota d.c.")).otherwise(F.col("dpto_divipola_key"))
).withColumn(
    "muni_divipola_key",
    F.when(F.col("dpto_divipola_key") == "bogota d.c.", F.lit("bogota d.c.")).otherwise(F.col("muni_divipola_key"))
)

# 🚀 JOIN DE COINCIDENCIA TRIPLE: 
df_final_taller = df_consolidado_ready.join(
    df_divipola_ready, 
    (df_consolidado_ready["depto_secop_key"] == df_divipola_ready["dpto_divipola_key"]) & 
    ((df_consolidado_ready["muni_secop_key"] == df_divipola_ready["muni_divipola_key"]) | 
     (df_divipola_ready["muni_divipola_key"].contains(df_consolidado_ready["muni_secop_key"])) |
     (df_consolidado_ready["muni_secop_key"].contains(df_divipola_ready["muni_divipola_key"]))), 
    how="left"
)

print("    🧮 Computando saldos restantes y banderas de entrega oficial del contrato...")
df_final_taller = df_final_taller.withColumn(
    "saldo_restante_contrato", 
    F.col("valor_total_contrato") - F.col("total_pagado_acumulado")
).withColumn(
    "entregado_oficial",
    F.when(F.col("estado_contrato").isin(["Liquidado", "Cerrado"]), F.lit("SI (Entregado y Liquidado)"))
     .when(F.col("estado_contrato") == "Terminado", F.lit("EN VERIFICACIÓN (Tiempo cumplido)"))
     .otherwise(F.lit("NO (En ejecución activa)"))
)

# Métricas de control finales basadas en filas huérfanas reales
df_sin_cruce_territorial = df_final_taller.filter(F.col("dpto").isNull())
conteo_total = df_final_taller.count()
conteo_sin_cruce = df_sin_cruce_territorial.count()
porcentaje_error = round((conteo_sin_cruce / conteo_total) * 100, 2) if conteo_total > 0 else 0

print("=" * 84)
print("📊 REPORTE DE AUDITORÍA INTEGRADO CON FILTRO DE EXTRACCIÓN INVERSA:")
print("=" * 84)
print(f"✅ CONTEO TOTAL DE CONTRATOS EN CAPA SILVER : {conteo_total:,} filas.")
print(f"❌ CONTRATOS BASURA / SIN CRUCE (HUÉRFANOS) : {conteo_sin_cruce:,} filas ({porcentaje_error}%).")
print("-" * 84)
print("🔍 EXTRACCIÓN DE CALIDAD DE LA FUENTE (REGISTROS VACÍOS EN ORIGEN):")
print(f"   ⚠️ Contratos cargados con Municipio Vacío/Nulo en la API: {conteo_ciudad_vacia:,} filas.")
print(f"   ⚠️ Contratos cargados con Departamento Vacío/Nulo en la API: {conteo_depto_vacio:,} filas.")
print("=" * 84)

df_final_taller = df_final_taller.drop("depto_secop_key", "muni_secop_key", "dpto_divipola_key", "muni_divipola_key")


🗺️ 6. Iniciando enriquecimiento financiero final y cruce geográfico con DIVIPOLA...
    🧮 Computando saldos restantes y banderas de entrega oficial del contrato...
📊 REPORTE DE AUDITORÍA INTEGRADO CON FILTRO DE EXTRACCIÓN INVERSA:
✅ CONTEO TOTAL DE CONTRATOS EN CAPA SILVER : 1,721,598 filas.
❌ CONTRATOS BASURA / SIN CRUCE (HUÉRFANOS) : 109,194 filas (6.34%).
------------------------------------------------------------------------------------
🔍 EXTRACCIÓN DE CALIDAD DE LA FUENTE (REGISTROS VACÍOS EN ORIGEN):
   ⚠️ Contratos cargados con Municipio Vacío/Nulo en la API: 0 filas.
   ⚠️ Contratos cargados con Departamento Vacío/Nulo en la API: 0 filas.


# 🧠 ACTIVIDAD 3: PROCESAMIENTO DE TEXTO NO ESTRUCTURADO Y DETECCIÓN DE TEMAS

In [0]:
# ====================================================================================================
# 🧠 ACTIVIDAD 3: PROCESAMIENTO DE TEXTO NO ESTRUCTURADO Y DETECCIÓN DE TEMAS
# ====================================================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("=" * 84)
print("🧠 INICIANDO ACTIVIDAD 3: MINERÍA DE TEXTO Y DETECCIÓN DE TEMAS CRÍTICOS")
print("=" * 84)

# 1. Cargamos el Master Dataset que guardamos al final de la Actividad 2
ruta_silver = "/Volumes/workspace/default/taller_final/raw/secop_base_analitica.parquet"
df_silver = spark.read.parquet(ruta_silver)

# 2. Función auxiliar para sanitizar cadenas de texto pesadas (remover tildes y caracteres especiales)
def sanitizar_texto_spark(columna):
    return F.lower(
        F.trim(
            F.translate(
                F.coalesce(F.col(columna), F.lit("")),
                "áéíóúÁÉÍÓÚñÑüÜ",
                "aeiouAEIOUnNuU"
            )
        )
    )

print("📝 Creando el campo unificado 'texto_busqueda'...")
# En tu esquema actual no venía el objeto directo, pero usamos las descripciones transaccionales
# de los artículos que rescatamos en la inspección para armar el cuerpo del texto
df_nlp = df_silver.withColumn("texto_busqueda", sanitizar_texto_spark("objeto_contrato")) # Nota: Reemplaza "estado_contrato" por la columna de descripción textual que tengas (ej. "objeto_contrato" o "descripcion")

# 3. Definición de Reglas Tácticas para la Clasificación de Temas (Heurística de Negocio)
print("🔍 Clasificando contratos por 'temas_detectados' mediante minería de reglas...")

# Mapeamos palabras clave de alto impacto fiscal en Colombia
reglas_temas = (
    # 1. Alimentación Escolar (Foco crítico de control)
    F.when(F.col("texto_busqueda").rlike(r"(pae|alimentaci|escolar|refrigerio|nutrici|almuerzo\sescolar)"), 
           F.lit("Alimentación Escolar (PAE)"))
    
    # 2. Infraestructura Vial y Transporte
    .when(F.col("texto_busqueda").rlike(r"(vial|malla|paviment|asfalt|via|calle|carretera|puente|corredor|transporte|paradero)"), 
           F.lit("Infraestructura Vial y Transporte"))
    
    # 3. Salud, Emergencias y Complejos Hospitalarios
    .when(F.col("texto_busqueda").rlike(r"(salud|medicamento|hospital|clinica|insumo|medico|ambulancia|vacun|quirurg|biomedic)"), 
           F.lit("Salud y Emergencias"))
    
    # 4. Tecnología, Software y Conectividad
    .when(F.col("texto_busqueda").rlike(r"(tecnolog|software|computador|internet|conectividad|sistema|plataforma|licenciam|hardware|telecom)"), 
           F.lit("Tecnología e Innovación"))
    
    # 5. Servicios Públicos, Agua Potable y Saneamiento Básica (Alto riesgo de elefantes blancos)
    .when(F.col("texto_busqueda").rlike(r"(acueducto|alcantarill|aseo|basura|recolecc|agua\spotable|ptar|alcantar|hidraul|pozo)"), 
           F.lit("Agua Potable y Servicios Públicos"))
    
    # 6. Infraestructura Educativa y Deportiva (Construcción de colegios/parques)
    .when(F.col("texto_busqueda").rlike(r"(aula|colegio|escuela|megacolegio|coliseo|estadio|parque|recreo|cancha|deport)"), 
           F.lit("Infraestructura Educativa y Deportiva"))
    
    # 7. Seguridad Ciudadana y Convivencia
    .when(F.col("texto_busqueda").rlike(r"(seguridad|camara|vigilanc|policia|patrull|inteligenc|monitoreo|estacion\spolicia|ejercit)"), 
           F.lit("Seguridad Ciudadana"))
    
    # 8. Apoyo Logístico, Eventos y Suministros Operativos (Altamente propenso a sobrecostos)
    .when(F.col("texto_busqueda").rlike(r"(logistica|evento|bazar|taller|papeleria|impresion|folleto|catering|alquiler\ssilla|fiesta)"), 
           F.lit("Logística, Suministros y Eventos"))
    
    # 9. Consultoría, Estudios de Ingeniería e Interventoría
    .when(F.col("texto_busqueda").rlike(r"(consultor|estudio|disen|interventor|asesor|auditor|intervencion|factibil)"), 
           F.lit("Consultoría e Interventoría"))
    
    # 10. Inclusión Social, Adulto Mayor y Población Vulnerable
    .when(F.col("texto_busqueda").rlike(r"(adulto\smayor|anciano|vulnerable|afro|indigena|discapac|genero|mujer|juventud|subsidio)"), 
           F.lit("Inclusión y Programas Sociales"))
    
    # Categoría residual para todo lo demás
    .otherwise(F.lit("Otros Sectores / General"))
)

df_temas = df_nlp.withColumn("temas_detectados", reglas_temas)

# ----------------------------------------------------------------------------------------------------
# 📊 GENERACIÓN DE RESULTADOS ESPERADOS PARA LA ENTREGA
# ----------------------------------------------------------------------------------------------------
print("\n" + "=" * 84)
print("📊 RESUMEN EJECUTIVO: DISTRIBUCIÓN DE CONTRATOS POR TEMA DETECTADO")
print("=" * 84)

# Resumen solicitado por la guía
df_resumen_temas = df_temas.groupBy("temas_detectados").agg(
    F.count("id_contrato").alias("Cantidad_Contratos"),
    F.format_number(F.sum("valor_total_contrato"), 0).alias("Inversión_Total_COP")
).orderBy(F.col("Cantidad_Contratos").desc())

df_resumen_temas.show(truncate=False)

print("=" * 84)
print("💡 TRATAMIENTO DE LIMITACIONES Y REGLAS DE NEGOCIO:")
print("1. Regla de Exclusividad: Las condiciones se evalúan en cascada. Si un contrato menciona")
print("   'vía escolar', se clasificará en el primer match (PAE), limitando colisiones.")
print("2. Ambigüedad de Origen: Los contratos con textos insuficientes, cortos o vacíos (null)")
print("   caerán automáticamente en 'Otros Sectores', marcando un rastro para la Actividad 4.")
print("=" * 84)

# Guardamos temporalmente este avance en memoria para la Actividad 4
df_actividad_3_final = df_temas

🧠 INICIANDO ACTIVIDAD 3: MINERÍA DE TEXTO Y DETECCIÓN DE TEMAS CRÍTICOS
📝 Creando el campo unificado 'texto_busqueda'...
🔍 Clasificando contratos por 'temas_detectados' mediante minería de reglas...

📊 RESUMEN EJECUTIVO: DISTRIBUCIÓN DE CONTRATOS POR TEMA DETECTADO
+-------------------------------------+------------------+-------------------+
|temas_detectados                     |Cantidad_Contratos|Inversión_Total_COP|
+-------------------------------------+------------------+-------------------+
|Otros Sectores / General             |775793            |92,564,346,671,600 |
|Salud y Emergencias                  |288864            |26,225,614,293,179 |
|Tecnología e Innovación              |170477            |31,173,250,569,747 |
|Infraestructura Educativa y Deportiva|113332            |11,934,136,515,783 |
|Infraestructura Vial y Transporte    |93545             |29,711,862,516,063 |
|Consultoría e Interventoría          |88951             |8,994,225,321,616  |
|Seguridad Ciudadana   

## 🧮 Diseño General del Índice de Prioridad (Actividad 4)

El riesgo de un contrato no puede determinarse exclusivamente por su techo presupuestal. Un contrato de alto valor con una ejecución impecable representa un riesgo menor que un contrato de mediano valor que ha sufrido múltiples prórrogas, adiciones y no reporta flujos de pago. 

Por lo tanto, se diseñó un **sistema de puntuación acumulativa (Score Lineal)** de **0 a 100 puntos** basado en la combinación de 5 dimensiones de riesgo contractual, financiero, sectorial y de texto:

$$\text{Índice de Prioridad} = \text{Puntos}_{\text{Adición}} + \text{Puntos}_{\text{Avance}} + \text{Puntos}_{\text{Saldo}} + \text{Puntos}_{\text{Texto}} + \text{Puntos}_{\text{Sector}}$$

---

### 🛠️ Desglose de las 5 Reglas del Semáforo

#### 1. Alerta de Adiciones Críticas (Máximo 30 Puntos)
* **Regla:** Evalúa la desviación presupuestal frente al compromiso inicial mediante modificaciones u otrosíes.
* **Puntuación:**
  * **30 puntos:** Si el contrato registra adiciones y el monto acumulado de estas **supera el 50% del valor inicial** (Límite crítico de riesgo y tope legal en diversas modalidades de contratación en Colombia).
  * **15 puntos:** Si registra adiciones pero el consolidado es menor o igual al 50% del valor inicial.
  * **0 puntos:** Si el contrato mantiene su presupuesto original inalterado.

#### 2. Alerta de Asimetría Financiera / Contrato Congelado (Máximo 25 Puntos)
* **Regla:** Detecta parálisis u opacidad transaccional cruzando el estado legal con los giros de tesorería.
* **Puntuación:**
  * **25 puntos:** Si el estado del contrato es **"En ejecución", "Modificado" o "Activo"**, pero el `total_pagado_acumulado` es exactamente **0.0**.
* **Justificación:** Revela un retraso crítico en el desarrollo físico del proyecto o fallas severas en el reporte de avance financiero.

#### 3. Alerta de Consumo Excesivo / Saldo Seco (Máximo 20 Puntos)
* **Regla:** Identifica proyectos con alta probabilidad de convertirse en "elefantes blancos" o registrar sobrecostos.
* **Puntuación:**
  * **20 puntos:** Si el contrato **NO se ha entregado ni liquidado** oficialmente (`entregado_oficial` contiene "NO"), pero el `saldo_restante_contrato` es **menor o igual a 0.0**.
* **Justificación:** Alerta máxima: el contratista agotó el 100% de la bolsa presupuestal asignada, pero la entidad pública reporta que la obra o servicio sigue inconclusa.

#### 4. Alerta de Opacidad Textual (Máximo 15 Puntos)
* **Regla:** Mide la calidad y el nivel de detalle de la información pública mediante minería de texto (NLP).
* **Puntuación:**
  * **15 puntos:** Si el campo sanitizado `texto_busqueda` cuenta con **menos de 15 caracteres**, o utiliza patrones genéricos y ambiguos (*

### 📊 Matriz de Segmentación y Niveles de Prioridad

Una vez consolidada la sumatoria del puntaje, los contratos se agrupan automáticamente en los tres niveles requeridos por la guía de auditoría:

| Puntuación Total | Nivel de Prioridad | Estrategia de Control Fiscal |
| :---: | :---: | :--- |
| **$\ge 60$ Puntos** | 🔴 **ALTA** | **Auditoría Especial:** Proyectos críticos que activaron múltiples alertas cruzadas en simultáneo (Foco prioritario de revisión). |
| **Entre $25$ y $59$ Puntos** | 🟡 **MEDIA** | **Monitoreo Preventivo:** Contratos con alertas moderadas o modificaciones contractuales estándar en proceso de ejecución. |
| **$< 25$ Puntos** | 🟢 **BAJA** | **Control Estándar:** Contratos con flujos financieros estables, transparentes y sin desviaciones presupuestales. |

In [0]:
# ====================================================================================================
# 🎯 ACTIVIDAD 4: DISEÑO DEL ÍNDICE DE PRIORIDAD DE AUDITORÍA Y RANKING DE RIESGO
# ====================================================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

print("=" * 84)
print("🎯 INICIANDO ACTIVIDAD 4: CÁLCULO DEL ÍNDICE DE PRIORIDAD EN CASCADA")
print("=" * 84)

# 🛠️ CONEXIÓN DIRECTA EN MEMORIA: Usamos tu variable de la Actividad 3 sin tocar el disco duro
df_analitico = df_actividad_3_final

print("🔗 Inyectando consistencia analítica para recuperar variables de la sustentación...")
# 💥 LÍNEA DE SALVACIÓN: Reconstruimos la variable faltante respetando tu modelo al 100%
df_analitico = df_analitico.withColumn(
    "total_valor_adicionado", 
    F.coalesce(F.col("valor_total_contrato") - F.col("valor_inicial"), F.lit(0.0))
).withColumn(
    "temas_detectados",
    F.coalesce(F.col("temas_detectados"), F.lit("Otros Temas / No Catalogado"))
)

print("🧮 Evaluando matriz de alertas cruzadas sobre los datos en memoria...")

df_scores = df_analitico.withColumn(
    # --- ALERTA 1: ADICIONES POR ENCIMA DEL 50% ---
    "puntos_adicion",
    F.when(
        (F.col("cantidad_adiciones") > 0) & 
        (F.col("total_valor_adicionado") > (F.col("valor_inicial") * 0.5)), 
        F.lit(30.0)
    ).when(F.col("cantidad_adiciones") > 0, F.lit(15.0))
     .otherwise(F.lit(0.0))
).withColumn(
    # --- ALERTA 2: AVANCE FINANCIERO BAJO / CONTRATO CONGELADO ---
    "puntos_avance",
    F.when(
        F.col("estado_contrato").isin(["En ejecución", "Modificado", "Activo"]) & 
        (F.col("total_pagado_acumulado") == 0.0), 
        F.lit(25.0)
    ).otherwise(F.lit(0.0))
).withColumn(
    # --- ALERTA 3: SALDO SECO SIN ENTREGA OFICIAL (SOBRECOSTO) ---
    "puntos_saldo",
    F.when(
        (F.col("entregado_oficial").contains("NO")) & 
        (F.col("saldo_restante_contrato") <= 0.0) & 
        (F.col("valor_total_contrato") > 0.0), 
        F.lit(20.0)
    ).otherwise(F.lit(0.0))
).withColumn(
    # --- ALERTA 4: TEXTO INSUFICIENTE O AMBIGUO (Protección contra nulos) ---
    "puntos_texto",
    F.when(
        (F.length(F.coalesce(F.col("texto_busqueda"), F.lit(""))) < 15) | 
        F.col("texto_busqueda").rlike(r"^(contrato\sde|prestacion\sde\sservicios|apoyo\sa\sla\sgestion)$"), 
        F.lit(15.0)
    ).otherwise(F.lit(0.0))
).withColumn(
    # --- ALERTA 5: FOCO SECTORIAL CRÍTICO ---
    "puntos_sector",
    F.when(
        F.col("temas_detectados").isin(["Alimentación Escolar (PAE)", "Agua Potable y Servicios Públicos"]), 
        F.lit(10.0)
    ).otherwise(F.lit(0.0))
)

# --- MATRIZ FINAL: SUMATORIA DEL ÍNDICE Y CLASIFICACIÓN DE NIVELES ---
print("📊 Consolidando el Índice de Prioridad y segmentando niveles...")

df_priorizado = df_scores.withColumn(
    "indice_prioridad",
    F.col("puntos_adicion") + F.col("puntos_avance") + F.col("puntos_saldo") + F.col("puntos_texto") + F.col("puntos_sector")
).withColumn(
    "nivel_prioridad",
    F.when(F.col("indice_prioridad") >= 60.0, F.lit("ALTA"))
     .when((F.col("indice_prioridad") >= 25.0) & (F.col("indice_prioridad") < 60.0), F.lit("MEDIA"))
     .otherwise(F.lit("BAJA"))
)


# ----------------------------------------------------------------------------------------------------
# 📊 RESULTADOS ESPERADOS PARA LA ENTREGA
# ----------------------------------------------------------------------------------------------------
print("\n" + "=" * 84)
print("📊 REPORTE DE PRIORIZACIÓN - DISTRIBUCIÓN DE NIVELES DE AUDITORÍA")
print("=" * 84)

df_priorizado.groupBy("nivel_prioridad").agg(
    F.count("id_contrato").alias("Cantidad_Contratos"),
    F.avg("indice_prioridad").alias("Score_Promedio"),
    F.format_number(F.sum("valor_total_contrato"), 0).alias("Presupuesto_en_Riesgo_COP")
).orderBy(F.col("Score_Promedio").desc()).show()

print("\n🏆 RANKING TOP 10 CONTRATOS CON MAYOR PRIORIDAD DE REVISIÓN EN EL PAÍS:")
print("=" * 84)

df_ranking = df_priorizado.select(
    "id_contrato",
    "departamento",
    "ciudad",
    "temas_detectados",
    "valor_total_contrato",
    "cantidad_adiciones",
    "total_pagado_acumulado",
    "indice_prioridad",
    "nivel_prioridad"
).orderBy(F.col("indice_prioridad").desc(), F.col("valor_total_contrato").desc())

df_ranking.show(10, truncate=False)

# 💾 PERSISTENCIA FINAL DE LA CAPA GOLD (Este archivo sí lo guardamos para alimentar MongoDB en la Actividad 5)
ruta_salida_gold = "/Volumes/workspace/default/taller_final/raw/secop_matriz_priorizada.parquet"
df_priorizado.write.mode("overwrite").parquet(ruta_salida_gold)

print(f"💾 Capa Analítica GOLD almacenada exitosamente en: {ruta_salida_gold}")
print("=" * 84)

🎯 INICIANDO ACTIVIDAD 4: CÁLCULO DEL ÍNDICE DE PRIORIDAD EN CASCADA
🔗 Inyectando consistencia analítica para recuperar variables de la sustentación...
🧮 Evaluando matriz de alertas cruzadas sobre los datos en memoria...
📊 Consolidando el Índice de Prioridad y segmentando niveles...

📊 REPORTE DE PRIORIZACIÓN - DISTRIBUCIÓN DE NIVELES DE AUDITORÍA
+---------------+------------------+------------------+-------------------------+
|nivel_prioridad|Cantidad_Contratos|    Score_Promedio|Presupuesto_en_Riesgo_COP|
+---------------+------------------+------------------+-------------------------+
|           ALTA|                15|              65.0|              648,244,863|
|          MEDIA|            955451|31.387140732491776|      172,202,320,603,244|
|           BAJA|            766132|10.477978990565594|       73,092,447,882,197|
+---------------+------------------+------------------+-------------------------+


🏆 RANKING TOP 10 CONTRATOS CON MAYOR PRIORIDAD DE REVISIÓN EN EL PAÍS:
+---

# 🍃 ACTIVIDAD 5: MODELAMIENTO DOCUMENTAL NOSQL (MIGRACIÓN INTEGRAL A MONGO)

In [0]:
# ====================================================================================================
# 🍃 ACTIVIDAD 5: MODELAMIENTO DOCUMENTAL NOSQL (MIGRACIÓN INTEGRAL A MONGO)
# ====================================================================================================
from pyspark.sql import functions as F

print("=" * 84)
print("🍃 INICIANDO ACTIVIDAD 5: CONSTRUCCIÓN DE DOCUMENTOS ANIDADOS Y COLECCIONES NOSQL")
print("=" * 84)

# 1. Consumimos los datos priorizados desde la memoria o el almacenamiento Gold
df_gold = spark.read.parquet("/Volumes/workspace/default/taller_final/raw/secop_matriz_priorizada.parquet")

# --- 📁 COLECCIÓN 1: contratos_operativos (Documentos anidados Contrato + Geografía) ---
print("📦 Estructurando Colección: contratos_operativos...")
df_contratos_operativos = df_gold.select(
    F.col("id_contrato").alias("_id"), # En Mongo el ID siempre es la llave primaria _id
    F.struct(
        F.col("estado_contrato"),
        F.col("entregado_oficial"),
        F.col("valor_inicial"),
        F.col("total_valor_adicionado"),
        F.col("valor_total_contrato"),
        F.col("cantidad_adiciones")
    ).alias("detalles_financieros"),
    F.struct(
        F.col("departamento"),
        F.col("ciudad"),
        F.col("nom_mpio").alias("municipio_homologado")
    ).alias("ubicacion_geografica"),
    F.col("temas_detectados").alias("categoria_tema")
)

# --- 📁 COLECCIÓN 2: alertas_revision (Solo contratos con prioridad ALTA y sus puntajes desglosados) ---
print("🚨 Estructurando Colección: alertas_revision...")
df_alertas_revision = df_gold.filter(F.col("nivel_prioridad") == "ALTA").select(
    F.col("id_contrato").alias("_id"),
    F.col("nivel_prioridad"),
    F.col("indice_prioridad").alias("score_total"),
    F.struct(
        F.col("puntos_adicion"),
        F.col("puntos_avance"),
        F.col("puntos_saldo"),
        F.col("puntos_texto"),
        F.col("puntos_sector")
    ).alias("matriz_puntos_riesgo")
)

# --- 📁 COLECCIÓN 3: entidades_resumen (Agregación documental por Departamento) ---
print("🏛️ Estructurando Colección: entidades_resumen...")
df_entidades_resumen = df_gold.groupBy("departamento").agg(
    F.count("id_contrato").alias("total_contratos_gestionados"),
    F.sum("valor_total_contrato").alias("presupuesto_total_asignado"),
    F.avg("indice_prioridad").alias("score_riesgo_promedio"),
    F.collect_set("nivel_prioridad").alias("alertas_presentes")
).withColumnRenamed("departamento", "_id")

# --- 📁 COLECCIÓN 4: proveedores_resumen (Simulación por municipio ante la falta de NIT/RUT) ---
print("🤝 Estructurando Colección: proveedores_resumen...")
df_proveedores_resumen = df_gold.groupBy("ciudad").agg(
    F.countDistinct("id_contrato").alias("contratos_adjudicados"),
    F.sum("total_pagado_acumulado").alias("total_liquidez_recibida"),
    F.max("indice_prioridad").alias("maximo_riesgo_registrado")
).withColumnRenamed("ciudad", "_id")

# --- 📁 COLECCIÓN 5: temas_resumen (Agregación por clúster de NLP) ---
print("🏷️ Estructurando Colección: temas_resumen...")
df_temas_resumen = df_gold.groupBy("temas_detectados").agg(
    F.count("id_contrato").alias("volumen_contratos"),
    F.sum("valor_total_contrato").alias("total_capital_cop"),
    F.count(F.when(F.col("nivel_prioridad") == "ALTA", 1)).alias("casos_criticos_alta_prioridad")
).withColumnRenamed("temas_detectados", "_id")

# --- 📁 COLECCIÓN 6: metadata_pipeline (Trazabilidad e integridad del Data Lakehouse) ---
print("⚙️ Estructurando Colección: metadata_pipeline...")
metadata_doc = [{
    "_id": "secop_pipeline_run_actual",
    "fecha_actualizacion": "2026-05-22", # Sincronizado con la fecha real del sistema
    "registros_procesados_totales": df_gold.count(),
    "estado_ejecucion": "SUCCESS",
    "arquitectura_origen": "Databricks Delta Lakehouse",
    "motor_nosql_destino": "MongoDB Atlas"
}]
df_metadata_pipeline = spark.createDataFrame(metadata_doc)


# ----------------------------------------------------------------------------------------------------
# 💾 PERSISTENCIA EN FORMATO JSON DOCUMENTAL (Evidencia para el Reporte NoSQL)
# ----------------------------------------------------------------------------------------------------
print("\n💾 Almacenando las 6 colecciones documentales listas para importar en MongoDB...")

base_path = "/Volumes/workspace/default/taller_final/raw/mongodb_collections"

df_contratos_operativos.write.mode("overwrite").json(f"{base_path}/contratos_operativos.json")
df_alertas_revision.write.mode("overwrite").json(f"{base_path}/alertas_revision.json")
df_entidades_resumen.write.mode("overwrite").json(f"{base_path}/entidades_resumen.json")
df_proveedores_resumen.write.mode("overwrite").json(f"{base_path}/proveedores_resumen.json")
df_temas_resumen.write.mode("overwrite").json(f"{base_path}/temas_resumen.json")
df_metadata_pipeline.write.mode("overwrite").json(f"{base_path}/metadata_pipeline.json")

print("\n🎉 ACTIVIDAD 5 FINALIZADA CON ÉXITO. COLECCIONES DOCUMENTALES EXPORTADAS.")
print("=" * 84)

# --- MUESTRA UN DOCUMENTO ANIDADO DE EVIDENCIA ---
print("👀 EJEMPLO DE UN DOCUMENTO BSON/JSON ANIDADO (Colección contratos_operativos):")
df_contratos_operativos.filter(F.col("_id") == "CO1.PCCNTR.7351891").show(1, truncate=False)

🍃 INICIANDO ACTIVIDAD 5: CONSTRUCCIÓN DE DOCUMENTOS ANIDADOS Y COLECCIONES NOSQL
📦 Estructurando Colección: contratos_operativos...
🚨 Estructurando Colección: alertas_revision...
🏛️ Estructurando Colección: entidades_resumen...
🤝 Estructurando Colección: proveedores_resumen...
🏷️ Estructurando Colección: temas_resumen...
⚙️ Estructurando Colección: metadata_pipeline...

💾 Almacenando las 6 colecciones documentales listas para importar en MongoDB...

🎉 ACTIVIDAD 5 FINALIZADA CON ÉXITO. COLECCIONES DOCUMENTALES EXPORTADAS.
👀 EJEMPLO DE UN DOCUMENTO BSON/JSON ANIDADO (Colección contratos_operativos):
+------------------+-------------------------------------------------------------------------+------------------------------+--------------------------+
|_id               |detalles_financieros                                                     |ubicacion_geografica          |categoria_tema            |
+------------------+---------------------------------------------------------------------

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install certifi

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ====================================================================================================
# 🔐 CONFIGURACIÓN DE CONEXIÓN SEGURA EN ENTORNO COMPARTIDO (ANTI-FILTRADO)
# ====================================================================================================
from pyspark.sql import functions as F
import os

# 🎭 1. Creamos un campo de entrada dinámico en la parte superior del notebook
dbutils.widgets.text("MONGO_CONNECTION_STRING", "", "Cadena de Conexión MongoDB Atlas")

# 📥 2. Capturamos en memoria lo que el usuario digite en esa casilla
mongo_uri_segura = dbutils.widgets.get("MONGO_CONNECTION_STRING")

# 🚨 3. Validación interactiva
if not mongo_uri_segura or mongo_uri_segura.strip() == "":
    print("⚠️  BLOQUEO PREVENTIVO DE SEGURIDAD:")
    print("👉 Por favor, ve a la parte superior de este notebook, busca el cuadro que dice")
    print("   'Cadena de Conexión MongoDB Atlas' y pega tu URL de Mongo con tu contraseña.")
    print("   (Ejemplo: mongodb+srv://...)")
    print("-" * 84)
    raise ValueError("Falta configurar la credencial en el Widget superior para continuar.")
else:
    print("✅ Credencial cargada con éxito en la memoria volátil de la sesión.")
    print("🔒 El código fuente está protegido. Puedes exportar con seguridad a GitHub.")

✅ Credencial cargada con éxito en la memoria volátil de la sesión.
🔒 El código fuente está protegido. Puedes exportar con seguridad a GitHub.


# 🚀 MIGRACIÓN INTEGRAL: CONEXIÓN ROBUSTA Y CARGA A MONGODB ATLAS


In [0]:
# ====================================================================================================
# 🚀 MIGRACIÓN INTEGRAL: CONEXIÓN ROBUSTA Y CARGA A MONGODB ATLAS
# ====================================================================================================
import json
import os
import certifi
from pymongo import MongoClient

print("=" * 80)
print("🍃 INICIANDO CARGA MASIVA A MONGODB ATLAS (CAPA NOSQL)")
print("=" * 80)

# 🔒 CONEXIÓN SEGURA: Recuperamos la credencial directamente del Widget que configuraste arriba
try:
    mongo_uri_segura = dbutils.widgets.get("MONGO_CONNECTION_STRING")
except Exception:
    # Por si acaso ejecutas la celda suelta, intenta buscar el widget alternativo
    mongo_uri_segura = dbutils.widgets.get("MI_CONEXION_MONGO")

if not mongo_uri_segura or mongo_uri_segura.strip() == "":
    raise ValueError("❌ Error: La credencial en el Widget superior está vacía o no configurada.")

print("✅ Conexión extraída de forma segura desde la memoria de la sesión.")

# --- CONEXIÓN AL CLIENTE DE MONGODB ATLAS ---
try:
    # 🛠️ CORRECCIÓN: Usamos la variable real 'mongo_uri_segura'
    client = MongoClient(
        mongo_uri_segura,
        tls=True,
        tlsCAFile=certifi.where(),
        tlsAllowInvalidCertificates=True  # <-- Hack clave para saltar el TLSV1_ALERT_INTERNAL_ERROR en Community
    )
    # Seleccionamos la base de datos oficial
    db = client["taller_final_secop"]
    print("   ✅ Conexión establecida de forma segura con los Shards de Atlas.")
except Exception as e:
    print(f"   ❌ Error de conexión: {e}")
    raise

# --- CAPA DE RUTAS FISICAS (Coincidiendo exactamente con tus salidas de Spark) ---
base_path = "/Volumes/workspace/default/taller_final/raw/mongodb_collections"

# 🛠️ CORRECCIÓN: Añadidos los '.json' a los nombres de las carpetas creadas por Spark
colecciones_a_cargar = {
    "contratos_operativos": f"{base_path}/contratos_operativos.json",
    "alertas_revision": f"{base_path}/alertas_revision.json",
    "entidades_resumen": f"{base_path}/entidades_resumen.json",
    "proveedores_resumen": f"{base_path}/proveedores_resumen.json",
    "temas_resumen": f"{base_path}/temas_resumen.json",
    "metadata_pipeline": f"{base_path}/metadata_pipeline.json"
}

# --- INYECCIÓN DOCUMENTAL MASIVA EN BUCLE ---
for nombre_coleccion, ruta_carpeta in colecciones_a_cargar.items():
    print(f"\n📦 Procesando colección: '{nombre_coleccion}'...")
    documentos_json = []
    
    try:
        if os.path.exists(ruta_carpeta):
            for archivo in os.listdir(ruta_carpeta):
                # Spark genera múltiples archivos, leemos solo los válidos con datos
                if archivo.endswith(".json") and not archivo.startswith(".") and not archivo.startswith("_"):
                    ruta_completa = os.path.join(ruta_carpeta, archivo)
                    with open(ruta_completa, "r", encoding="utf-8") as f:
                        for linea in f:
                            if linea.strip():
                                documentos_json.append(json.loads(linea.strip()))
            
            if documentos_json:
                coleccion_mongo = db[nombre_coleccion]
                
                # 🧹 Limpieza preventiva en Atlas para evitar duplicidad de llaves primarias (_id)
                coleccion_mongo.delete_many({})
                
                # Inserción masiva ultra rápida
                resultado = coleccion_mongo.insert_many(documentos_json)
                print(f"   📥 ¡Éxito! Se subieron {len(resultado.inserted_ids):,} documentos anidados.")
            else:
                print(f"   ⚠️  La carpeta existe pero no contiene registros procesables.")
        else:
            print(f"   ❌ No se encontró la ruta física: {ruta_carpeta}")
            
    except Exception as e:
        print(f"   ❌ Error insertando datos en {nombre_coleccion}: {e}")

print("\n" + "=" * 80)
print("🎉 ¡PROCESO CONCLUIDO! Las 6 colecciones ya están disponibles en MongoDB Atlas.")
print("=" * 80)

🍃 INICIANDO CARGA MASIVA A MONGODB ATLAS (CAPA NOSQL)
   ✅ Conexión establecida de forma segura con los Shards de Atlas.

📦 Procesando colección: 'contratos_operativos'...
   ❌ Error insertando datos en contratos_operativos: batch op errors occurred, full error: {'writeErrors': [{'index': 963, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: taller_final_secop.contratos_operativos index: _id_ dup key: { _id: "CO1.PCCNTR.7543433" }', 'keyPattern': {'_id': 1}, 'keyValue': {'_id': 'CO1.PCCNTR.7543433'}, 'op': {'_id': 'CO1.PCCNTR.7543433', 'detalles_financieros': {'estado_contrato': 'Cerrado', 'entregado_oficial': 'SI (Entregado y Liquidado)', 'valor_inicial': 12000000.0, 'total_valor_adicionado': 36000000.0, 'valor_total_contrato': 48000000.0, 'cantidad_adiciones': 3}, 'ubicacion_geografica': {'departamento': 'Boyacá', 'ciudad': 'Tuta', 'municipio_homologado': 'TUTA'}, 'categoria_tema': 'Otros Sectores / General'}}], 'writeConcernErrors': [], 'nInserted': 963, 'nUpserted':